# 声音克隆教程

本教程介绍声音克隆技术，包括：

1. **声音克隆原理** - 从参考音频提取说话人特征
2. **说话人编码器** - 提取说话人嵌入
3. **GE2E 损失** - 训练说话人编码器
4. **声音克隆流程** - 零样本语音合成
5. **实践应用** - 模型创建与使用

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 声音克隆原理

### 1.1 什么是声音克隆？

声音克隆是指从少量参考音频中提取说话人的声音特征，然后用这些特征生成具有相同音色的新语音。

```
参考音频 → [说话人编码器] → 说话人嵌入
                              ↓
文本 → [TTS 模型] ←──────────┘
           ↓
      克隆的语音
```

### 1.2 主要方法

| 方法 | 优点 | 缺点 |
|------|------|------|
| 说话人编码器 | 零样本，无需微调 | 相似度有限 |
| 微调 | 高质量 | 需要更多数据 |
| 适配器 | 平衡质量和数据需求 | 需要少量训练 |

In [ ]:
# 可视化声音克隆流程
fig, ax = plt.subplots(figsize=(14, 6))

# 绘制模块
modules = [
    ("参考音频\n(3-10秒)", 1, 5, "lightyellow"),
    ("说话人\n编码器", 1, 3.5, "lightgreen"),
    ("说话人嵌入\n(256维向量)", 1, 2, "lightblue"),
    ("输入文本", 4, 5, "lightyellow"),
    ("TTS 模型", 4, 3.5, "lightcoral"),
    ("克隆语音", 4, 2, "plum"),
]

for name, x, y, color in modules:
    ax.add_patch(plt.Rectangle((x-0.5, y-0.4), 1, 0.8, 
                                fill=True, color=color, edgecolor='black'))
    ax.text(x, y, name, ha='center', va='center', fontsize=10)

# 箭头
arrows = [(1, 4.6, 1, 3.9), (1, 3.1, 1, 2.4), (4, 4.6, 4, 3.9), (4, 3.1, 4, 2.4)]
for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
               arrowprops=dict(arrowstyle='->', color='black'))

# 说话人嵌入到 TTS 的连接
ax.annotate('', xy=(3.5, 3.5), xytext=(1.5, 2),
           arrowprops=dict(arrowstyle='->', color='blue', lw=2,
                          connectionstyle='arc3,rad=0.3'))

ax.set_xlim(0, 5.5)
ax.set_ylim(1, 6)
ax.set_title('声音克隆流程', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. 说话人编码器

说话人编码器从参考音频中提取固定维度的说话人嵌入向量。

In [ ]:
from voice_cloning import SpeakerEncoderConfig, SpeakerEncoder, create_speaker_encoder

config = SpeakerEncoderConfig()
print("说话人编码器配置:")
print(f"  n_mels: {config.n_mels}")
print(f"  hidden_size: {config.hidden_size}")
print(f"  num_layers: {config.num_layers}")
print(f"  embedding_size: {config.embedding_size}")

In [ ]:
# 创建说话人编码器
encoder = create_speaker_encoder("base")

# 模拟输入: Mel 频谱
mel = torch.randn(2, 80, 200)  # [batch, n_mels, time]
lengths = torch.tensor([200, 150])

# 提取说话人嵌入
embedding = encoder(mel, lengths)
print(f"输入 Mel: {mel.shape}")
print(f"说话人嵌入: {embedding.shape}")
print(f"嵌入范数: {embedding.norm(dim=-1).tolist()}")

## 3. GE2E 损失

GE2E (Generalized End-to-End) 损失用于训练说话人编码器，使同一说话人的嵌入相似，不同说话人的嵌入不同。

In [ ]:
from voice_cloning import GE2ELoss

ge2e_loss = GE2ELoss()

# 模拟: 4个说话人，每人3条语音
speakers_per_batch = 4
utterances_per_speaker = 3
embedding_size = 256

embeddings = torch.randn(speakers_per_batch * utterances_per_speaker, embedding_size)
embeddings = F.normalize(embeddings, p=2, dim=-1)

loss = ge2e_loss(embeddings, speakers_per_batch, utterances_per_speaker)
print(f"GE2E 损失: {loss.item():.4f}")

## 4. 声音克隆流程

In [ ]:
from voice_cloning import VoiceCloner, SpeakerAdapter, VoiceCloningConfig

# 创建说话人编码器
speaker_encoder = create_speaker_encoder("base")

# 模拟参考音频
reference_mel = torch.randn(1, 80, 300)

# 提取说话人嵌入
with torch.no_grad():
    speaker_embedding = speaker_encoder(reference_mel)

print(f"参考音频: {reference_mel.shape}")
print(f"说话人嵌入: {speaker_embedding.shape}")

In [ ]:
# 说话人相似度比较
mel1 = torch.randn(1, 80, 200)
mel2 = torch.randn(1, 80, 200)
mel3 = mel1 + torch.randn_like(mel1) * 0.1  # 相似的音频

with torch.no_grad():
    emb1 = speaker_encoder(mel1)
    emb2 = speaker_encoder(mel2)
    emb3 = speaker_encoder(mel3)

sim_12 = F.cosine_similarity(emb1, emb2).item()
sim_13 = F.cosine_similarity(emb1, emb3).item()

print(f"音频1 vs 音频2 (不同): {sim_12:.4f}")
print(f"音频1 vs 音频3 (相似): {sim_13:.4f}")

## 5. 实践应用

### 5.1 创建不同大小的编码器

In [ ]:
for size in ["tiny", "base", "large"]:
    encoder = create_speaker_encoder(size)
    num_params = sum(p.numel() for p in encoder.parameters())
    print(f"{size:>6} 编码器参数量: {num_params / 1e6:.2f}M")

## 总结

### 声音克隆的关键技术

1. **说话人编码器**: 从音频提取说话人特征
2. **GE2E 损失**: 对比学习训练编码器
3. **说话人适配器**: 将嵌入注入 TTS 模型
4. **零样本克隆**: 无需微调即可克隆新声音

### 应用场景

- 个性化语音助手
- 有声书制作
- 语音翻译保留原声
- 虚拟主播